# 📊 마인드맵 실험 결과 분석

## 개요
- **목적**: Claude/GPT-4 실험 결과 CSV 파일을 업로드하여 자동 분석
- **기능**: 통계 요약, 시각화, 표 생성, 결과 다운로드
- **입력**: `experiments.csv` 또는 `experiments_gpt.csv`

## 사용 방법
1. ✅ 패키지 설치
2. ✅ 라이브러리 import
3. 📁 CSV 파일 업로드
4. 📊 자동 분석 실행
5. 💾 결과 다운로드

---
## 1️⃣ 패키지 설치

In [ ]:
# 필수 패키지 설치
!pip install pandas matplotlib seaborn numpy scipy -q

print("✅ 패키지 설치 완료")

---
## 2️⃣ 라이브러리 Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter
from google.colab import files
import json
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✅ 라이브러리 import 완료")

---
## 3️⃣ CSV 파일 업로드

**업로드할 파일**:
- Claude 실험: `experiments.csv`
- GPT-4 실험: `experiments_gpt.csv`

아래 셀을 실행하고 파일을 선택하세요!

In [ ]:
# 파일 업로드
print("📁 CSV 파일을 선택하세요...")
uploaded = files.upload()

# 업로드된 파일명 확인
if len(uploaded) == 0:
    print("❌ 파일이 업로드되지 않았습니다.")
else:
    filename = list(uploaded.keys())[0]
    print(f"\n✅ 파일 업로드 완료: {filename}")
    
    # API 구분
    if 'gpt' in filename.lower():
        api_name = "GPT-4"
    else:
        api_name = "Claude"
    
    print(f"   API: {api_name}")

---
## 4️⃣ 데이터 로드 및 검증

In [ ]:
# CSV 파일 로드
try:
    df = pd.read_csv(filename)
    print(f"✅ 데이터 로드 완료")
    print(f"   총 실험: {len(df)}개")
    print(f"   컬럼 수: {len(df.columns)}개")
    print(f"\n📋 컬럼 목록:")
    for col in df.columns:
        print(f"   - {col}")
except Exception as e:
    print(f"❌ 파일 로드 실패: {e}")

---
## 5️⃣ 기본 통계 요약

In [ ]:
print("="*80)
print(f"📊 {api_name} 실험 결과 요약")
print("="*80)

# 성공/실패 통계
total = len(df)
success = len(df[df['status'] == 'success'])
failed = len(df[df['status'] == 'failed'])
success_rate = (success / total * 100) if total > 0 else 0

print(f"\n🎯 전체 통계:")
print(f"   총 실험: {total}개")
print(f"   성공: {success}개 ({success_rate:.1f}%)")
print(f"   실패: {failed}개 ({100-success_rate:.1f}%)")

# 성공한 실험만 필터링
df_success = df[df['status'] == 'success'].copy()

if len(df_success) > 0:
    print(f"\n📈 점수 통계 (성공한 실험만):")
    print(f"   평균 점수: {df_success['total_score'].mean():.2f}")
    print(f"   표준편차: {df_success['total_score'].std():.2f}")
    print(f"   중앙값: {df_success['total_score'].median():.2f}")
    print(f"   최고 점수: {df_success['total_score'].max():.2f}")
    print(f"   최저 점수: {df_success['total_score'].min():.2f}")
    
    print(f"\n⏱️ 실행 시간 통계:")
    print(f"   평균: {df_success['execution_time_sec'].mean():.2f}초")
    print(f"   중앙값: {df_success['execution_time_sec'].median():.2f}초")
    print(f"   최대: {df_success['execution_time_sec'].max():.2f}초")
    print(f"   최소: {df_success['execution_time_sec'].min():.2f}초")
else:
    print("\n⚠️ 성공한 실험이 없어 점수 통계를 표시할 수 없습니다.")

# 실패 분석
if failed > 0:
    print(f"\n❌ 실패 분석:")
    df_failed = df[df['status'] == 'failed']
    error_counts = Counter(df_failed['error'].dropna())
    print(f"   에러 유형: {len(error_counts)}가지\n")
    for error, count in error_counts.most_common(3):
        percentage = (count / failed) * 100
        error_short = error[:80] + '...' if len(error) > 80 else error
        print(f"   [{count:3d}회 / {percentage:5.1f}%] {error_short}")

print("\n" + "="*80)

---
## 6️⃣ 시각화 - 성공/실패 분포

In [ ]:
# 성공/실패 파이 차트
fig, ax = plt.subplots(figsize=(8, 6))

sizes = [success, failed]
labels = [f'Success\n{success} ({success_rate:.1f}%)', 
          f'Failed\n{failed} ({100-success_rate:.1f}%)']
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)

ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, explode=explode, textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title(f'{api_name} Experiment Success Rate\n(Total: {total} experiments)', 
             fontsize=14, weight='bold', pad=20)

plt.tight_layout()
plt.savefig('success_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 그래프 저장: success_rate.png")

---
## 7️⃣ 시각화 - 점수 분포 (성공한 실험만)

In [ ]:
if len(df_success) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 히스토그램
    axes[0].hist(df_success['total_score'], bins=20, color='#3498db', 
                 edgecolor='black', alpha=0.7)
    axes[0].axvline(df_success['total_score'].mean(), color='red', 
                    linestyle='--', linewidth=2, label=f"Mean: {df_success['total_score'].mean():.2f}")
    axes[0].axvline(df_success['total_score'].median(), color='green', 
                    linestyle='--', linewidth=2, label=f"Median: {df_success['total_score'].median():.2f}")
    axes[0].set_xlabel('Total Score', fontsize=12, weight='bold')
    axes[0].set_ylabel('Frequency', fontsize=12, weight='bold')
    axes[0].set_title(f'{api_name} Score Distribution', fontsize=14, weight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 박스플롯
    bp = axes[1].boxplot(df_success['total_score'], vert=True, patch_artist=True,
                         boxprops=dict(facecolor='#3498db', alpha=0.7),
                         medianprops=dict(color='red', linewidth=2),
                         whiskerprops=dict(linewidth=1.5),
                         capprops=dict(linewidth=1.5))
    axes[1].set_ylabel('Total Score', fontsize=12, weight='bold')
    axes[1].set_title(f'{api_name} Score Box Plot', fontsize=14, weight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # 통계 정보 추가
    stats_text = f"Mean: {df_success['total_score'].mean():.2f}\n"
    stats_text += f"Median: {df_success['total_score'].median():.2f}\n"
    stats_text += f"Std: {df_success['total_score'].std():.2f}"
    axes[1].text(1.15, df_success['total_score'].mean(), stats_text,
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('score_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ 그래프 저장: score_distribution.png")
else:
    print("⚠️ 성공한 실험이 없어 점수 분포를 그릴 수 없습니다.")

---
## 8️⃣ 시각화 - 조합별 성능

In [ ]:
if len(df_success) > 0:
    # 조합별 평균 점수 계산
    combination_scores = df_success.groupby('combination_id')['total_score'].agg(['mean', 'std', 'count']).reset_index()
    combination_scores = combination_scores.sort_values('mean', ascending=False)
    
    # 조합 이름 매핑
    comb_names = df_success.groupby('combination_id')['combination_name'].first().to_dict()
    combination_scores['name'] = combination_scores['combination_id'].map(comb_names)
    
    # 바 차트
    fig, ax = plt.subplots(figsize=(14, 8))
    
    bars = ax.barh(range(len(combination_scores)), combination_scores['mean'], 
                   xerr=combination_scores['std'], capsize=5,
                   color=plt.cm.viridis(np.linspace(0.3, 0.9, len(combination_scores))),
                   edgecolor='black', linewidth=0.5)
    
    ax.set_yticks(range(len(combination_scores)))
    ax.set_yticklabels([f"{row['combination_id']}\n{row['name']}" 
                        for _, row in combination_scores.iterrows()], fontsize=9)
    ax.set_xlabel('Average Total Score', fontsize=12, weight='bold')
    ax.set_title(f'{api_name} Performance by Algorithm Combination\n(sorted by score)', 
                fontsize=14, weight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='x')
    
    # 점수 값 표시
    for i, (idx, row) in enumerate(combination_scores.iterrows()):
        ax.text(row['mean'] + 0.5, i, f"{row['mean']:.2f}", 
               va='center', fontsize=9, weight='bold')
    
    plt.tight_layout()
    plt.savefig('combination_performance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ 그래프 저장: combination_performance.png")
else:
    print("⚠️ 성공한 실험이 없어 조합별 성능을 그릴 수 없습니다.")

---
## 9️⃣ 시각화 - 대화 유형별 성능

In [ ]:
if len(df_success) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # 대화 유형별 박스플롯
    conversation_types = df_success['conversation_type'].unique()
    data_by_type = [df_success[df_success['conversation_type'] == ct]['total_score'].values 
                    for ct in conversation_types]
    
    bp = ax.boxplot(data_by_type, labels=conversation_types, patch_artist=True)
    
    # 색상 설정
    colors = plt.cm.Set3(np.linspace(0, 1, len(conversation_types)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_xlabel('Conversation Type', fontsize=12, weight='bold')
    ax.set_ylabel('Total Score', fontsize=12, weight='bold')
    ax.set_title(f'{api_name} Performance by Conversation Type', 
                fontsize=14, weight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 평균값 표시
    means = [df_success[df_success['conversation_type'] == ct]['total_score'].mean() 
             for ct in conversation_types]
    ax.plot(range(1, len(conversation_types)+1), means, 'ro-', linewidth=2, 
           markersize=8, label='Mean')
    ax.legend()
    
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('conversation_type_performance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ 그래프 저장: conversation_type_performance.png")
else:
    print("⚠️ 성공한 실험이 없어 대화 유형별 성능을 그릴 수 없습니다.")

---
## 🔟 상세 표 - 조합별 통계

In [ ]:
if len(df_success) > 0:
    print("="*100)
    print(f"📊 조합별 상세 통계")
    print("="*100)
    
    combination_stats = df_success.groupby('combination_id').agg({
        'total_score': ['mean', 'std', 'min', 'max'],
        'execution_time_sec': 'mean',
        'combination_name': 'first'
    }).round(2)
    
    combination_stats.columns = ['Mean Score', 'Std Score', 'Min Score', 'Max Score', 'Avg Time (s)', 'Name']
    combination_stats = combination_stats.sort_values('Mean Score', ascending=False)
    
    print(combination_stats.to_string())
    print("\n" + "="*100)
    
    # CSV 저장
    combination_stats.to_csv('combination_statistics.csv')
    print("\n✅ 표 저장: combination_statistics.csv")
else:
    print("⚠️ 성공한 실험이 없어 통계표를 생성할 수 없습니다.")

---
## 1️⃣1️⃣ 상세 표 - 대화 유형별 통계

In [ ]:
if len(df_success) > 0:
    print("="*100)
    print(f"📊 대화 유형별 상세 통계")
    print("="*100)
    
    conversation_stats = df_success.groupby('conversation_type').agg({
        'total_score': ['mean', 'std', 'min', 'max', 'count'],
        'execution_time_sec': 'mean'
    }).round(2)
    
    conversation_stats.columns = ['Mean Score', 'Std Score', 'Min Score', 'Max Score', 'Count', 'Avg Time (s)']
    conversation_stats = conversation_stats.sort_values('Mean Score', ascending=False)
    
    print(conversation_stats.to_string())
    print("\n" + "="*100)
    
    # CSV 저장
    conversation_stats.to_csv('conversation_statistics.csv')
    print("\n✅ 표 저장: conversation_statistics.csv")
else:
    print("⚠️ 성공한 실험이 없어 통계표를 생성할 수 없습니다.")

---
## 1️⃣2️⃣ Top & Bottom 조합

In [ ]:
if len(df_success) > 0:
    print("="*100)
    print(f"🏆 최고 성능 조합 (Top 5)")
    print("="*100)
    
    top_combinations = combination_scores.head(5)
    for rank, (idx, row) in enumerate(top_combinations.iterrows(), 1):
        print(f"\n#{rank}. {row['combination_id']} - {row['name']}")
        print(f"     평균 점수: {row['mean']:.2f} ± {row['std']:.2f}")
        print(f"     실험 수: {int(row['count'])}개")
    
    print("\n" + "="*100)
    print(f"📉 최저 성능 조합 (Bottom 5)")
    print("="*100)
    
    bottom_combinations = combination_scores.tail(5)
    for rank, (idx, row) in enumerate(bottom_combinations.iterrows(), 1):
        print(f"\n#{rank}. {row['combination_id']} - {row['name']}")
        print(f"     평균 점수: {row['mean']:.2f} ± {row['std']:.2f}")
        print(f"     실험 수: {int(row['count'])}개")
    
    print("\n" + "="*100)
else:
    print("⚠️ 성공한 실험이 없어 순위를 표시할 수 없습니다.")

---
## 1️⃣3️⃣ 결과 다운로드

생성된 모든 그래프와 통계표를 다운로드합니다.

In [ ]:
import os

print("📦 생성된 파일 목록:\n")

output_files = [
    'success_rate.png',
    'score_distribution.png',
    'combination_performance.png',
    'conversation_type_performance.png',
    'combination_statistics.csv',
    'conversation_statistics.csv'
]

existing_files = [f for f in output_files if os.path.exists(f)]

for f in existing_files:
    print(f"   ✅ {f}")

if len(existing_files) > 0:
    print(f"\n💾 {len(existing_files)}개 파일을 다운로드합니다...")
    for f in existing_files:
        files.download(f)
    print("\n✅ 다운로드 완료!")
else:
    print("\n⚠️ 다운로드할 파일이 없습니다.")

---
## 🎉 분석 완료!

### 생성된 파일:
1. **그래프**:
   - `success_rate.png` - 성공/실패 분포
   - `score_distribution.png` - 점수 분포
   - `combination_performance.png` - 조합별 성능
   - `conversation_type_performance.png` - 대화 유형별 성능

2. **통계표**:
   - `combination_statistics.csv` - 조합별 통계
   - `conversation_statistics.csv` - 대화 유형별 통계

### 다음 단계:
- 다운로드한 파일을 보고서에 첨부
- Claude/GPT-4 결과를 각각 분석한 후 비교
- `analyze_comparison.py`로 통합 비교 분석